# docking-surrogate-al: GPU smoke test

Runs the whole pipeline on 200 molecules **on a CUDA GPU**, to verify the code
paths that could not be exercised on the development machine:

* `run.device: cuda` resolution and the startup assertion
* mixed precision (autocast + GradScaler), which is auto-disabled on CPU
* batched MC-dropout on GPU
* GPU fields in the run manifest (name, VRAM, CUDA version, peak memory, samples/s)
* the `dock` stage against a real smina binary

**These are not results.** 200 molecules at docking exhaustiveness 1 exists to
exercise code cheaply. Do not quote any number this notebook prints.

Requires: Accelerator = **GPU T4 x2** (or P100), and **Internet = On**.


In [ ]:
# 1. What hardware did we actually get?
!nvidia-smi
import torch, platform
print("torch:", torch.__version__)
print("torch CUDA build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  {p.total_memory/1024**2:.0f} MiB  capability {p.major}.{p.minor}")
print("python:", platform.python_version())
import os; print("cpu cores:", os.cpu_count())


In [ ]:
# 2. Clone the repository.
import os, shutil
if os.path.exists("/kaggle/working/docking-surrogate-al"):
    shutil.rmtree("/kaggle/working/docking-surrogate-al")
!git clone --depth 1 https://github.com/Joana-Mansa/docking-surrogate-al.git /kaggle/working/docking-surrogate-al
%cd /kaggle/working/docking-surrogate-al
!git log --oneline -1


In [ ]:
# 3. Python dependencies.
#
# Kaggle's image already ships a CUDA build of torch, and we deliberately keep
# it rather than installing the cu128 pin from pyproject.toml: reinstalling
# torch on Kaggle is slow and risks a driver mismatch. torch-geometric is pure
# Python and dispatches to whatever torch build is present, so the GPU paths
# under test are unaffected. The exact versions are recorded in the manifest.
!pip install -q torch_geometric rdkit
import torch_geometric, rdkit
print("torch_geometric:", torch_geometric.__version__)
print("rdkit:", rdkit.__version__)


In [ ]:
# 4. smina (CPU-only docking binary) via micromamba.
#
# smina is not on PyPI. micromamba is a single static binary, which is the
# least fragile way to pull a conda-forge package inside a Kaggle session.
import os, subprocess
if not os.path.exists("/kaggle/working/bin/micromamba"):
    !mkdir -p /kaggle/working/bin
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /kaggle/working bin/micromamba
# micromamba, unlike conda, will NOT create a prefix implicitly on
# `install`; it requires `create` first. Using `install` on a missing
# prefix fails with "No prefix found" and leaves smina uninstalled.
!/kaggle/working/bin/micromamba create -y -p /kaggle/working/mamba-env -c conda-forge smina openbabel 2>&1 | tail -5

os.environ["PATH"] = "/kaggle/working/mamba-env/bin:" + os.environ["PATH"]
print(subprocess.run(["which","smina"], capture_output=True, text=True).stdout)
print(subprocess.run(["smina","--version"], capture_output=True, text=True).stdout.splitlines()[:2])


In [ ]:
# 5. Run the pipeline on the GPU.
#
# Only the device and pin_memory are overridden; everything else is the
# committed smoke config, so this is the same pipeline CI runs on CPU.
# run.name stays "smoke" so the AMP parity check below finds docked.csv.
import os, time
os.environ["PYTHONPATH"] = "src"

OVERRIDES = "--set run.device=cuda --set train.pin_memory=true"


def stage(name, extra=""):
    t0 = time.time()
    cmd = f"python -m docking_surrogate_al.cli {name} --config configs/smoke.yaml {OVERRIDES} {extra}"
    rc = os.system(cmd)
    took = time.time() - t0
    print("")
    print(f"--- {name}: exit {rc} in {took:.0f}s ---")
    print("")
    assert rc == 0, f"stage {name} failed with exit {rc}"
    return took


timings = {}
for st in ["prepare-data", "dock", "train", "evaluate", "run-al"]:
    timings[st] = stage(st)
print("stage timings (s):", {k: round(v) for k, v in timings.items()})

In [ ]:
# 6. What did the manifest record about the GPU?
import json
m = json.load(open("runs/smoke/manifest.json"))

print("hardware:")
for k, v in m["hardware"].items():
    print(f"  {k}: {v}")

print("")
print("per-stage timing / throughput / peak VRAM:")
for name, s in m["stages"].items():
    print(f"  {name}: wall={s.get('wall_clock_s')}s "
          f"samples/s={s.get('samples_per_second')} "
          f"peak_alloc={s.get('peak_allocated_mb')}MiB "
          f"peak_reserved={s.get('peak_reserved_mb')}MiB")

print("")
print("docking:", m.get("docking"))
print("config_hash:", m["config_hash"])
print("git commit:", m["git"]["commit"])

# These assertions are the point of the notebook: a silent CPU fallback
# must fail the run rather than produce a green notebook proving nothing.
assert m["hardware"]["device_type"] == "cuda", "did not actually run on the GPU"
assert m["hardware"]["gpu_name"], "no GPU name recorded"
print("")
print("OK: ran on", m["hardware"]["gpu_name"])

In [ ]:
# 7. Confirm mixed precision was genuinely active (not silently disabled).
import json
m = json.load(open("runs/smoke/manifest.json"))
amp_flags = {k: v.get("amp_enabled") for k, v in m["training"].items()}
print("amp_enabled per split:", amp_flags)
assert all(amp_flags.values()), "AMP was disabled; expected it active on CUDA"
print("OK: autocast + GradScaler were active during training.")

In [ ]:
# 8. AMP parity check: is the AMP-vs-FP32 gap inside seed variance?
#
# This is the check the README promises and could not run without a GPU.
# Loose by design: it tests "within seed noise", not bit-identity.
!python scripts/check_amp_parity.py --config configs/smoke.yaml --seeds 3 --epochs 25


In [ ]:
# 9. Results summary and figures.
import glob
import pandas as pd
from IPython.display import display, Image

print("=== metrics (smoke numbers, NOT results) ===")
display(pd.read_csv("runs/smoke/metrics.csv").head(30))

print("=== active learning, final round ===")
al = pd.read_csv("runs/smoke/active_learning.csv")
final = al[al.round_index == al.round_index.max()]
display(final.groupby("strategy")["top_k_recall"].agg(["mean", "std"]))

for path in sorted(glob.glob("figures/*.png")):
    print(path)
    display(Image(path))

In [ ]:
# 10. Bundle the outputs so they can be downloaded from the session.
!cd /kaggle/working/docking-surrogate-al && tar czf /kaggle/working/smoke_gpu_outputs.tar.gz runs/smoke figures
!ls -lh /kaggle/working/smoke_gpu_outputs.tar.gz
print("Download this from the Kaggle session's Output panel.")
